# 07 — RAG Retrieval Engine

This notebook validates the semantic RAG pipeline built in **07b_Vector_RAG.ipynb**.

### What changed from the old ID-based approach:
| | Old (ID-based) | New (Semantic) |
|---|---|---|
| Matching | hadm_id lookup in 1,261 summaries | ChromaDB cosine similarity across all summaries |
| Coverage | 1,192 / 4,508 (26.4%) | 4,508 / 4,508 (100%) |
| Context | Clinical summary only | Summary + top-3 similar historical cases |
| Output file | `rag_prompts.json` | `rag_prompts_semantic.json` |

**This notebook does not retrain the model.** It loads `model_artifacts.pkl` from notebook 05.

In [ ]:
import json
import os
import numpy as np
import pandas as pd

BASE_DIR     = os.path.dirname(os.path.abspath('__file__'))
PROMPTS_PATH = os.path.join(BASE_DIR, '..', 'rag_prompts_semantic.json')
PROMPTS_PATH = os.path.normpath(PROMPTS_PATH)

assert os.path.exists(PROMPTS_PATH), f"ERROR: {PROMPTS_PATH} not found — run 07b_Vector_RAG.ipynb first"

with open(PROMPTS_PATH) as f:
    rag_prompts = json.load(f)

print(f"Loaded {len(rag_prompts)} semantic RAG prompts from rag_prompts_semantic.json")
print(f"Keys in each record: {list(rag_prompts[0].keys())}")
print(f"\nCoverage: {len(rag_prompts)} / 4508 patients ({100*len(rag_prompts)/4508:.1f}%)")

In [2]:
# ── Step 1: Load ALL patients from DB and extract features ───────────────────
print("Connecting to DB to load model_features...")
con = get_db_connection()
df = con.execute("SELECT * FROM model_features").fetchdf()
con.close()
print(f"Loaded {len(df)} total patients from model_features.")

# Add lab CHANGE features (last - first)
LABS = [
    'creatinine', 'urea_nitrogen', 'sodium', 'potassium', 'glucose',
    'hemoglobin', 'white_blood_cells', 'platelet_count', 'bicarbonate',
    'calcium_total', 'inrpt', 'ptt', 'troponin_t',
    'creatine_kinase_mb_isoenzyme'
]
for lab in LABS:
    df[f'{lab}_change'] = df[f'{lab}_last'] - df[f'{lab}_first']

# Fill missing numerical values with median
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for c in ['readmitted_30d', 'hadm_id', 'subject_id']:
    if c in num_cols:
        num_cols.remove(c)
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill and encode categorical columns
cat_cols = ['gender', 'admission_type', 'insurance', 'marital_status', 'race', 'discharge_location']
for col in cat_cols:
    df[col] = df[col].fillna('UNKNOWN')

df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Separate features, labels, and id indices
all_hadm_ids = df_encoded['hadm_id'].values
X_all = df_encoded.drop(columns=['subject_id', 'hadm_id', 'readmitted_30d'])
y_all = df_encoded['readmitted_30d']

print(f"Feature matrix shape (all patients): {X_all.shape}")

Connecting to DB to load model_features...
Connected to DuckDB at: d:\Capstone project\Capstone_Healthcare_Decision_Intelligence_Agent\dataset\hf_project.duckdb
Loaded 4508 total patients from model_features.
Feature matrix shape (all patients): (4508, 137)


In [ ]:
import pickle

# Load the model trained in notebook 05 — DO NOT retrain here
artifacts_path = os.path.normpath(os.path.join(BASE_DIR, '..', 'model_artifacts.pkl'))
assert os.path.exists(artifacts_path), "ERROR: model_artifacts.pkl not found — run notebook 05 first"

with open(artifacts_path, 'rb') as f:
    artifacts = pickle.load(f)

print(f"Model type     : {artifacts.get('model_type', 'unknown')}")
print(f"Model class    : {type(artifacts['model']).__name__}")
print(f"Features (k)   : {len(artifacts['selected_features'])}")
print(f"Medication features in top {len(artifacts['selected_features'])}:")
MED_KEYWORDS = ['loop','ace_arb','beta','aldo','digoxin','anticoag','unique_drugs','furosemide','gdmt']
for f in artifacts['selected_features']:
    if any(k in f for k in MED_KEYWORDS):
        print(f"  ✓ {f}")

In [ ]:
# ── Risk score distribution from semantic prompts ────────────────────────────
risk_scores = [p['risk_pct'] for p in rag_prompts]
risk_arr    = np.array(risk_scores)

high   = sum(r >= 60 for r in risk_arr)
medium = sum(30 <= r < 60 for r in risk_arr)
low    = sum(r < 30 for r in risk_arr)

print("Risk score distribution across all 4,508 patients:")
print(f"  HIGH   (≥60%) : {high:,}  ({100*high/len(risk_arr):.1f}%)")
print(f"  MEDIUM (30-60%): {medium:,}  ({100*medium/len(risk_arr):.1f}%)")
print(f"  LOW    (<30%)  : {low:,}   ({100*low/len(risk_arr):.1f}%)")
print(f"\n  Mean risk  : {risk_arr.mean():.1f}%")
print(f"  Max risk   : {risk_arr.max():.1f}%")
print(f"  Min risk   : {risk_arr.min():.1f}%")

In [ ]:
# ── Sample 3 prompts to validate the semantic context ────────────────────────
# Show one HIGH, one MEDIUM, one LOW risk patient

samples = {
    'HIGH':   next((p for p in rag_prompts if p['risk_pct'] >= 60), None),
    'MEDIUM': next((p for p in rag_prompts if 30 <= p['risk_pct'] < 60), None),
    'LOW':    next((p for p in rag_prompts if p['risk_pct'] < 30), None),
}

for label, p in samples.items():
    if p is None:
        continue
    print('=' * 70)
    print(f"[{label} RISK]  hadm_id: {p['hadm_id']}  |  risk: {p['risk_pct']}%")
    print('=' * 70)
    # Show first 600 chars of the prompt context
    print(p['prompt_context'][:600])
    print('...')
    print()

In [ ]:
# ── Validate similar_cases are embedded in each prompt ───────────────────────
# Each prompt should contain similar historical cases from ChromaDB

sample = rag_prompts[0]
ctx    = sample['prompt_context']

has_similar = '--- SIMILAR HISTORICAL' in ctx
print(f"Similar cases section present : {has_similar}")
print(f"Prompt length (chars)         : {len(ctx)}")
print(f"Similar cases (raw list)      : {len(sample.get('similar_cases', []))} entries")
print()

# Count prompts that have meaningful similar cases section
with_similar = sum(1 for p in rag_prompts if '--- SIMILAR HISTORICAL' in p['prompt_context'])
print(f"Prompts with similar cases : {with_similar:,} / {len(rag_prompts):,} ({100*with_similar/len(rag_prompts):.1f}%)")
print()
print("RAG pipeline validation PASSED — semantic prompts are ready for notebook 08.")

In [ ]:
# ── Final pipeline summary ────────────────────────────────────────────────────
llm_outputs_path = os.path.normpath(os.path.join(BASE_DIR, '..', 'llm_outputs_semantic.json'))
llm_done = 0
if os.path.exists(llm_outputs_path):
    with open(llm_outputs_path) as f:
        llm_data = json.load(f)
    llm_done = sum(1 for r in llm_data if 'error' not in r)

print("=" * 55)
print("RAG PIPELINE SUMMARY")
print("=" * 55)
print(f"  Semantic prompts generated : {len(rag_prompts):,}  (100% coverage)")
print(f"  LLM outputs completed      : {llm_done:,} / {len(rag_prompts):,}  ({100*llm_done/len(rag_prompts):.1f}%)")
print(f"  Remaining for notebook 08  : {len(rag_prompts) - llm_done:,}")
print()
print("Next step → Run 08_RAG_LLM_Generation.ipynb to generate doctor alerts.")